# Stage 1 Idiom Signal Model Comparison

This dashboard provides persistent, epoch-wise convergence analysis and aggregate validation performance tracking across the modernized AdMIRe stage-1 idiom signal architectures.


## Comparative Setup & Architectural Details

Each model formulation represents a specific strategy for encoding target expressions and structural context. The pipeline evaluates raw target embeddings, marked token sequences, LLM perturbation difference vectors, and advanced span-pooling fusions.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
PROJECT_ROOT = ROOT if (ROOT / "signal_models").exists() else ROOT.parent
STAGE1_DIR = PROJECT_ROOT / "signal_models"

MODEL_DIRS = [
    "baseline_mlp",
    "lang_marked_mlp",
    "perturbation_mlp",
    "lang_marked_perturbation_mlp",
    "target_perturbation_mlp",
]

MODEL_LABELS = {
    "baseline_mlp": "Baseline MLP (Raw Target)",
    "lang_marked_mlp": "Marked MLP ([LANG] + [MWE])",
    "perturbation_mlp": "Perturbation MLP (LLM Diffs)",
    "lang_marked_perturbation_mlp": "Fusion MLP (Marked + Perturbation)",
    "target_perturbation_mlp": "Target Span-Pool + Perturbation MLP",
}

MODEL_DETAILS = {
    "baseline_mlp": {
        "input": "Raw target sentence embedding",
        "head": "MLP",
        "signal_type": "Text-only baseline",
        "description": "Uses sentence-level target representations without explicit MWE markers or perturbation signals.",
    },
    "lang_marked_mlp": {
        "input": "Marked sentence embedding ([LANG] + <mwe>)",
        "head": "MLP",
        "signal_type": "Marked text",
        "description": "Marks target expressions with specific boundary and language tokens to guide attention.",
    },
    "perturbation_mlp": {
        "input": "LLM perturbation difference vectors",
        "head": "MLP",
        "signal_type": "Perturbation-only",
        "description": "Leverages semantic differences between target sentences and LLM-generated variations.",
    },
    "lang_marked_perturbation_mlp": {
        "input": "Marked embedding + perturbation diffs",
        "head": "MLP",
        "signal_type": "Context + perturbation",
        "description": "Fuses contextual token boundary markers with structural LLM perturbation difference features.",
    },
    "target_perturbation_mlp": {
        "input": "Target Span-Pool + Perturbation Diffs",
        "head": "MLP",
        "signal_type": "Span-Pool + perturbation",
        "description": "Extracts precise MWE representations via token-offset span pooling fused with LLM perturbation differences.",
    },
}

# Configure beautiful premium aesthetics
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlepad": 12,
    "font.size": 11,
})

print("Environment and styling configured successfully.")


## Model Inputs & Architecture Overview

Summary of the encoding formulations utilized across the candidate models.


In [ ]:
input_rows = []
for m_dir in MODEL_DIRS:
    details = MODEL_DETAILS[m_dir]
    input_rows.append({
        "Architecture": MODEL_LABELS[m_dir],
        "Signal Type": details["signal_type"],
        "Input Representation": details["input"],
        "Classifier Head": details["head"],
        "Formulation Rationale": details["description"],
    })
df_inputs = pd.DataFrame(input_rows)
display(df_inputs.style.set_properties(subset=["Formulation Rationale"], **{"text-align": "left"}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'left'), ('padding', '8px')]}
]))


## Aggregate Performance Outputs

Loading serialized validation metrics from each model's standard execution cache.


In [ ]:
def load_json(path: Path) -> dict:
    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)
    return {}

rows = []
per_run_rows = []
hyper_rows = []

for model_dir in MODEL_DIRS:
    output_dir = STAGE1_DIR / model_dir / "outputs_xlm_base"
    if not output_dir.exists():
        output_dir = STAGE1_DIR / model_dir / "outputs"
    
    metrics = load_json(output_dir / "metrics.json")
    config = load_json(output_dir / "config.json")
    details = MODEL_DETAILS[model_dir]

    run_metrics = []
    for item in metrics.get("per_seed", []):
        best_metrics = item.get("best_metrics", {})
        run_metrics.append(best_metrics)
        per_run_rows.append({
            "model": model_dir,
            "label": MODEL_LABELS[model_dir],
            "seed": item.get("seed"),
            "best_val_f1": item.get("best_val_f1", best_metrics.get("f1")),
            "best_epoch": item.get("best_epoch"),
            "accuracy": best_metrics.get("accuracy"),
            "f1": best_metrics.get("f1"),
        })

    run_df = pd.DataFrame(run_metrics)
    mean_acc = run_df["accuracy"].mean() if not run_df.empty and "accuracy" in run_df else np.nan
    f1_range = run_df["f1"].max() - run_df["f1"].min() if not run_df.empty and "f1" in run_df else np.nan

    aggregate = {
        "model": model_dir,
        "label": MODEL_LABELS[model_dir],
        "signal_type": details["signal_type"],
        "mean_val_f1": metrics.get("mean_val_f1", run_df["f1"].mean() if not run_df.empty and "f1" in run_df else np.nan),
        "best_val_f1": metrics.get("best_val_f1", run_df["f1"].max() if not run_df.empty and "f1" in run_df else np.nan),
        "best_seed": metrics.get("best_seed"),
        "best_epoch_avg": pd.DataFrame(per_run_rows).loc[lambda d: d["model"] == model_dir, "best_epoch"].mean() if per_run_rows else np.nan,
        "mean_accuracy": mean_acc,
        "f1_range": f1_range,
    }
    rows.append(aggregate)

    hyper = metrics.get("hyperparameters", config.get("hyperparameters", {}))
    hyper_rows.append({"Architecture": MODEL_LABELS[model_dir], **hyper})

summary_df = pd.DataFrame(rows).sort_values("mean_val_f1", ascending=False).reset_index(drop=True)
per_run_df = pd.DataFrame(per_run_rows)
hyper_df = pd.DataFrame(hyper_rows)

display_cols = [
    "label", "signal_type", "mean_val_f1", "best_val_f1", "mean_accuracy", "f1_range", "best_epoch_avg"
]
col_renames = {
    "label": "Architecture", "signal_type": "Signal Type", "mean_val_f1": "Mean Val F1",
    "best_val_f1": "Best Val F1", "mean_accuracy": "Mean Val Accuracy", "f1_range": "F1 Range Across Seeds",
    "best_epoch_avg": "Avg Optimal Epoch"
}

styled_summary = summary_df[display_cols].rename(columns=col_renames).style.format({
    "Mean Val F1": "{:.4f}",
    "Best Val F1": "{:.4f}",
    "Mean Val Accuracy": "{:.4f}",
    "F1 Range Across Seeds": "{:.4f}",
    "Avg Optimal Epoch": "{:.1f}",
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', 'white'), ('font-weight', 'bold'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '8px')]}
]).highlight_max(subset=["Mean Val F1", "Best Val F1", "Mean Val Accuracy"], color="#d1fae5")

display(styled_summary)


## Visual Profile: Validation F1 Convergence Ranking

Comparative visualization depicting optimal and mean F1 scores achieved across different signal embedding methodologies.


In [ ]:
plot_df = summary_df.sort_values("mean_val_f1", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

y_positions = np.arange(len(plot_df))
bar_height = 0.35

color_mean = "#475569" 
color_best = "#94a3b8" 
winner_mean = "#4f46e5" 
winner_best = "#06b6d4"  

means = plot_df["mean_val_f1"].values
bests = plot_df["best_val_f1"].values
labels = plot_df["label"].values

for i in range(len(plot_df)):
    is_winner = labels[i] == summary_df.iloc[0]["label"]
    c_m = winner_mean if is_winner else color_mean
    c_b = winner_best if is_winner else color_best
    
    ax.barh(y_positions[i] - bar_height/2, means[i], height=bar_height, color=c_m, alpha=0.9, edgecolor="none", label="Mean Val F1" if i==len(plot_df)-1 else "")
    ax.barh(y_positions[i] + bar_height/2, bests[i], height=bar_height, color=c_b, alpha=0.9, edgecolor="none", label="Best Val F1" if i==len(plot_df)-1 else "")
    
    ax.text(means[i] + 0.006, y_positions[i] - bar_height/2, f"{means[i]:.4f}", va="center", fontsize=10, fontweight="bold", color=c_m)
    ax.text(bests[i] + 0.006, y_positions[i] + bar_height/2, f"{bests[i]:.4f}", va="center", fontsize=10, fontweight="bold", color=c_b)

ax.set_yticks(y_positions)
ax.set_yticklabels(labels, fontsize=11, fontweight="bold", color="#1e293b")
ax.set_xlim(0.5, 1.02)
ax.set_xlabel("Validation F1 Score", fontsize=12, fontweight="bold", color="#1e293b", labelpad=12)
ax.set_title("Stage-1 Idiom Signal Optimization: F1 Evaluation Ranking", fontsize=15, fontweight="bold", color="#0f172a", pad=18)

ax.grid(False)

sns.despine(left=True, bottom=False)
ax.spines["bottom"].set_color("#cbd5e1")
ax.spines["bottom"].set_linewidth(1.2)

ax.legend(loc="lower right", frameon=True, facecolor="#f8fafc", edgecolor="#e2e8f0", framealpha=1.0, fontsize=10)

plt.tight_layout()
plt.show()

## Selected Optimization Hyperparameters

Configuration profile tracking model capacity and hyperparameter setup across variants.


In [ ]:
display(hyper_df.fillna("-").style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', 'white'), ('font-weight', 'bold'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '8px'), ('text-align', 'center')]}
]))


## Delta Analysis: Margin Over Baselines

Quantifying performance gains achieved relative to standard text-only baselines and perturbation-only approaches.


In [ ]:
base_row = summary_df[summary_df["model"] == "baseline_mlp"]
pert_row = summary_df[summary_df["model"] == "perturbation_mlp"]

base_f1 = base_row["mean_val_f1"].values[0] if not base_row.empty else 0.0
pert_f1 = pert_row["mean_val_f1"].values[0] if not pert_row.empty else 0.0
best_f1 = summary_df["mean_val_f1"].max()

delta_rows = []
for idx, row in summary_df.iterrows():
    delta_rows.append({
        "Architecture": row["label"],
        "Mean Val F1": row["mean_val_f1"],
        "Gain vs Baseline": row["mean_val_f1"] - base_f1,
        "Gain vs Perturbation": row["mean_val_f1"] - pert_f1,
        "Distance to Optimal": best_f1 - row["mean_val_f1"],
    })

df_delta = pd.DataFrame(delta_rows)
display(df_delta.style.format({
    "Mean Val F1": "{:.4f}",
    "Gain vs Baseline": "{:+,.4f}",
    "Gain vs Perturbation": "{:+,.4f}",
    "Distance to Optimal": "{:.4f}",
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', 'white'), ('font-weight', 'bold'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '8px')]}
]))


## Final Architectural Decision

Comprehensive multi-dimensional evaluation isolates the top-performing idiomaticity encoding framework.


In [ ]:
winner = summary_df.iloc[0]
print("="*60)
print(f"★ OPTIMAL STAGE-1 ARCHITECTURE SELECTED: {winner['label']} ★")
print("="*60)
print(f"Mean Validation F1 Score : {winner['mean_val_f1']:.4f}")
print(f"Peak Validation F1 Score : {winner['best_val_f1']:.4f}")
print(f"Mean Validation Accuracy : {winner['mean_accuracy']:.4f}")
print(f"Signal Methodology       : {winner['signal_type']}")
print("-"*60)
print(f"Formulation Design       : {MODEL_DETAILS[winner['model']]['description']}")
print("="*60)


## Optimal Architecture Training Dynamics: Epoch-Wise Trajectories

In [ ]:
def load_history(path: Path) -> list:
    hist_file = path / "history.json"
    if hist_file.exists():
        with hist_file.open("r", encoding="utf-8") as f:
            return json.load(f)
    return []

winner_dir = winner["model"]
out_dir = STAGE1_DIR / winner_dir / "outputs_xlm_base"
if not out_dir.exists():
    out_dir = STAGE1_DIR / winner_dir / "outputs"

history = load_history(out_dir)

if history:
    df_h = pd.DataFrame(history)
    epochs = df_h["epoch"]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Premium vibrant styling
    ax.plot(epochs, df_h["train_loss"], label="Training Loss", color="#2563eb", linewidth=2.5, marker="o")
    ax.plot(epochs, df_h["val_loss"], label="Validation Loss", color="#10b981", linewidth=2.5, linestyle="--", marker="s")
    
    # Highlight minimum validation loss
    best_ep_loss = df_h.loc[df_h["val_loss"].idxmin()]
    ax.scatter(best_ep_loss["epoch"], best_ep_loss["val_loss"], color="#ef4444", s=140, zorder=5, label=f"Min Val Loss ({best_ep_loss['val_loss']:.3f})")
    
    # Set a focused vertical scale with fixed major tick intervals of 0.1 to emphasize the convergence drop
    ax.set_yticks(np.arange(0.0, 1.1, 0.1))
    ax.set_ylim(0.0, 1.0)
    ax.set_xlim(epochs.min() - 0.2, epochs.max() + 0.2)
    
    ax.set_title(f"Convergence Trajectory: {winner['label']}", fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Training Epoch", fontweight="bold", labelpad=10)
    ax.set_ylabel("Cross-Entropy Loss", fontweight="bold", labelpad=10)
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=1.0)
    ax.grid(True, linestyle=":", alpha=0.6)
    
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print(f"No cached history.json found for the optimal architecture ({winner['label']}).")

## Discriminative Analysis: Idiomatic vs. Literal Signal Distributions

To visually inspect the real-world discriminative capability of our optimal architecture (**Target Span-Pool + Perturbation MLP**), we extract its raw output confidence scores (sigmoid probabilities) across all samples. 

A high-performing representation space should naturally push **Literal** expressions towards $0.0$ and **Idiomatic** expressions towards $1.0$, minimizing standard deviation overlap around the $0.5$ decision boundary.

In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve essential dependencies and absolute paths
matched_file = STAGE1_DIR / "data/matched_records.json"
cache_file = STAGE1_DIR / "cache/cache_xlm_base_target_pert.pt"
weights_file = STAGE1_DIR / winner["model"] / "outputs_xlm_base/idiom_signal.pt"

if not weights_file.exists():
    weights_file = STAGE1_DIR / winner["model"] / "outputs/idiom_signal.pt"

if matched_file.exists() and cache_file.exists() and weights_file.exists():
    # 1. Load dataset payload to extract evaluation targets
    with matched_file.open("r", encoding="utf-8") as f:
        records = json.load(f)
    labels = np.array([float(r["label"]) for r in records])
    
    # 2. Reconstruct input space via cached continuous span features
    pooled = torch.load(cache_file, map_location="cpu").float()
    t = pooled[:, 0]
    features = torch.cat([
        t, 
        torch.abs(t - pooled[:, 1]),
        torch.abs(t - pooled[:, 2]),
        torch.abs(t - pooled[:, 3]),
        torch.abs(t - pooled[:, 4]),
    ], dim=-1)
    
    # 3. Restore trained classifier parameters and topological topology
    ckpt = torch.load(weights_file, map_location="cpu")
    input_dim = ckpt["input_dim"]
    hp = ckpt["hyperparameters"]
    
    classifier = nn.Sequential(
        nn.Linear(input_dim, hp["hidden_1"]), nn.BatchNorm1d(hp["hidden_1"]), nn.GELU(), nn.Dropout(hp["dropout_1"]),
        nn.Linear(hp["hidden_1"], hp["hidden_2"]), nn.BatchNorm1d(hp["hidden_2"]), nn.GELU(), nn.Dropout(hp["dropout_2"]),
        nn.Linear(hp["hidden_2"], 1),
    )
    classifier.load_state_dict(ckpt["classifier_state_dict"])
    classifier.eval()
    
    # 4. Extract raw empirical confidence distributions
    with torch.no_grad():
        logits = classifier(features).squeeze(-1)
        probs = torch.sigmoid(logits).numpy()
        
    # Isolate sample predictions per underlying class context
    idiom_probs = probs[labels == 1]
    literal_probs = probs[labels == 0]
    
    mean_id, std_id = idiom_probs.mean(), idiom_probs.std()
    mean_lit, std_lit = literal_probs.mean(), literal_probs.std()
    
    print("="*65)
    print("★ SIGNAL DISCRIMINATION STATISTICAL PROFILE ★")
    print("="*65)
    print(f"Idiomatic Class (Label=1) : Mean = {mean_id:.4f} | Std Dev = {std_id:.4f}")
    print(f"Literal Class   (Label=0) : Mean = {mean_lit:.4f} | Std Dev = {std_lit:.4f}")
    print(f"Centroid Separation Margin: {mean_id - mean_lit:.4f}")
    print("="*65)
    
    # 5. Render high-fidelity comparative density profiles
    fig, ax = plt.subplots(figsize=(10, 5), facecolor="#ffffff")
    ax.set_facecolor("#ffffff")
    
    # Premium contrasting layout aesthetics
    c_idiom = "#10b981"   # Vibrant Emerald Green
    c_literal = "#ef4444" # Deep Coral Red
    
    # Render localized Kernel Density Estimations
    sns.kdeplot(literal_probs, fill=True, color=c_literal, alpha=0.35, linewidth=2.5, label=f"Literal (Mean: {mean_lit:.2f} ± {std_lit:.2f})", ax=ax)
    sns.kdeplot(idiom_probs, fill=True, color=c_idiom, alpha=0.35, linewidth=2.5, label=f"Idiomatic (Mean: {mean_id:.2f} ± {std_id:.2f})", ax=ax)
    
    # Demarcate operational classification bound
    ax.axvline(x=0.5, color="#64748b", linestyle="--", linewidth=1.8, label="Decision Boundary (0.5)")
    
    # Typographical parameters
    ax.set_xlim(-0.05, 1.05)
    ax.set_xlabel("Model Confidence Score $P(Idiom)$", fontsize=12, fontweight="bold", color="#1e293b", labelpad=12)
    ax.set_ylabel("Density", fontsize=12, fontweight="bold", color="#1e293b", labelpad=12)
    ax.set_title("Idiomatic vs. Literal Signal Separation Distribution", fontsize=14, fontweight="bold", color="#0f172a", pad=15)
    
    ax.grid(False)
    sns.despine(left=True)
    ax.spines["bottom"].set_color("#cbd5e1")
    ax.spines["bottom"].set_linewidth(1.2)
    
    ax.legend(loc="upper center", frameon=True, facecolor="#f8fafc", edgecolor="#e2e8f0", framealpha=1.0, fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Skipped distribution analysis due to missing cached features or model parameter archives.")